In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Algoritmul cuantic de optimizare aproximativă

*Estimare de utilizare: 22 de minute pe un procesor Heron r3 (NOTĂ: Aceasta este doar o estimare. Durata ta de execuție poate varia.)*
## Context
Acest tutorial demonstrează cum să implementezi **Algoritmul Cuantic de Optimizare Aproximativă (QAOA)** – o metodă iterativă hibridă (cuantică-clasică) – în contextul pattern-urilor Qiskit. Mai întâi vei rezolva problema **Tăieturii Maxime** (sau **Max-Cut**) pentru un graf mic, apoi vei învăța cum să o execuți la scară de utilitate. Toate execuțiile pe hardware din tutorial ar trebui să se încadreze în limita de timp a planului Open, accesibil gratuit.

Problema Max-Cut este o problemă de optimizare dificil de rezolvat (mai precis, este o problemă *NP-hard*), cu diverse aplicații în clustering, știința rețelelor și fizica statistică. Acest tutorial consideră un graf de noduri conectate prin muchii și urmărește să împartă nodurile în două mulțimi astfel încât numărul de muchii traversate de această tăietură să fie maximizat.

![Ilustrarea unei probleme max-cut](../docs/images/tutorials/quantum-approximate-optimization-algorithm/maxcut-illustration.avif)
## Cerințe
Înainte de a începe acest tutorial, asigură-te că ai instalate următoarele:
- Qiskit SDK v1.0 sau mai recent, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 sau mai recent (`pip install qiskit-ibm-runtime`)

În plus, vei avea nevoie de acces la o instanță pe [IBM Quantum Platform](/guides/cloud-setup). Reține că acest tutorial nu poate fi executat pe planul Open, deoarece rulează sarcini de lucru folosind [sesiuni](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session), care sunt disponibile doar cu acces la planul Premium.
## Configurare

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence


from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Partea I. QAOA la scară mică
Prima parte a acestui tutorial folosește o problemă Max-Cut de dimensiuni mici pentru a ilustra pașii necesari rezolvării unei probleme de optimizare cu un calculator cuantic.

Pentru a oferi un context înainte de a mapa problema la un algoritm cuantic, poți înțelege mai bine cum devine problema Max-Cut o problemă de optimizare combinatorie clasică, considerând mai întâi minimizarea unei funcții $f(x)$

$$
\min_{x\in {0, 1}^n}f(x),
$$

unde intrarea $x$ este un vector ale cărui componente corespund fiecărui nod al unui graf. Apoi, constrânge fiecare componentă să fie fie $0$, fie $1$ (reprezentând includerea sau neincluderea în tăietură). Acest exemplu de scară mică folosește un graf cu $n=5$ noduri.

Poți scrie o funcție pentru o pereche de noduri $i,j$ care indică dacă muchia corespunzătoare $(i,j)$ face parte din tăietură. De exemplu, funcția $x_i + x_j - 2 x_i x_j$ este 1 doar dacă unul dintre $x_i$ sau $x_j$ este 1 (ceea ce înseamnă că muchia face parte din tăietură) și zero în caz contrar. Problema maximizării muchiilor din tăietură poate fi formulată ca

$$
\max_{x\in {0, 1}^n} \sum_{(i,j)} x_i + x_j - 2 x_i x_j,
$$

care poate fi rescrisă ca o minimizare de forma

$$
\min_{x\in {0, 1}^n} \sum_{(i,j)}  2 x_i x_j - x_i - x_j.
$$

Minimul lui $f(x)$ în acest caz apare atunci când numărul de muchii traversate de tăietură este maxim. După cum poți vedea, nu există nimic legat de calculul cuantic până acum. Trebuie să reformulezi această problemă în ceva pe care un calculator cuantic îl poate înțelege.
Inițializează problema ta creând un graf cu $n=5$ noduri.

In [2]:
n = 5

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n, 1))
edge_list = [
    (0, 1, 1.0),
    (0, 2, 1.0),
    (0, 4, 1.0),
    (1, 2, 1.0),
    (2, 3, 1.0),
    (3, 4, 1.0),
]
graph.add_edges_from(edge_list)
draw_graph(graph, node_size=600, with_labels=True)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif)

### Pasul 1: Maparea intrărilor clasice la o problemă cuantică
Primul pas al pattern-ului este de a mapa problema clasică (graful) în **circuite** și **operatori** cuantici. Pentru aceasta, există trei pași principali:

1. Utilizarea unei serii de reformulări matematice pentru a reprezenta această problemă folosind notația problemelor de Optimizare Binară Pătratică Fără Constrângeri (QUBO).
2. Rescrierea problemei de optimizare ca un Hamiltonian al cărui stare fundamentală corespunde soluției care minimizează funcția de cost.
3. Crearea unui Circuit cuantic care va pregăti starea fundamentală a acestui Hamiltonian printr-un proces similar cu recoacerea cuantică.

**Notă:** În metodologia QAOA, vrei în cele din urmă să ai un operator (**Hamiltonian**) care reprezintă **funcția de cost** a algoritmului nostru hibrid, precum și un Circuit parametrizat (**Ansatz**) care reprezintă stări cuantice cu soluții candidate la problemă. Poți eșantiona din aceste stări candidate și apoi le poți evalua folosind funcția de cost.

#### Graf &rarr; problemă de optimizare
Primul pas al mapării este o schimbare de notație. Cele ce urmează exprimă problema în notație QUBO:

$$
\min_{x\in {0, 1}^n}x^T Q x,
$$

unde $Q$ este o matrice $n\times n$ de numere reale, $n$ corespunde numărului de noduri din graful tău, $x$ este vectorul variabilelor binare introduse mai sus, iar $x^T$ indică transpusa vectorului $x$.

```
Maximize
 -2*x_0*x_1 - 2*x_0*x_2 - 2*x_0*x_4 - 2*x_1*x_2 - 2*x_2*x_3 - 2*x_3*x_4 + 3*x_0
 + 2*x_1 + 3*x_2 + 2*x_3 + 2*x_4

Subject to
  No constraints

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4
```
### Problemă de optimizare &rarr; Hamiltonian
Poți apoi reformula problema QUBO ca un **Hamiltonian** (aici, o matrice care reprezintă energia unui sistem):

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

<details>
<summary>
**Pași de reformulare din problema QAOA la Hamiltonian**
</summary>

Pentru a demonstra cum poate fi rescrisă problema QAOA în acest mod, înlocuiește mai întâi variabilele binare $x_i$ cu un nou set de variabile $z_i\in{-1, 1}$ prin

$$
x_i = \frac{1-z_i}{2}.
$$

Aici poți vedea că dacă $x_i$ este $0$, atunci $z_i$ trebuie să fie $1$. Când $x_i$-urile sunt substituite de $z_i$-uri în problema de optimizare ($x^TQx$), se poate obține o formulare echivalentă.

$$
x^TQx=\sum_{ij}Q_{ij}x_ix_j \\ =\frac{1}{4}\sum_{ij}Q_{ij}(1-z_i)(1-z_j) \\=\frac{1}{4}\sum_{ij}Q_{ij}z_iz_j-\frac{1}{4}\sum_{ij}(Q_{ij}+Q_{ji})z_i + \frac{n^2}{4}.
$$

Acum, dacă definim $b_i=-\sum_{j}(Q_{ij}+Q_{ji})$, eliminăm prefactorul și termenul constant $n^2$, ajungem la cele două formulări echivalente ale aceleiași probleme de optimizare.

$$
\min_{x\in{0,1}^n} x^TQx\Longleftrightarrow \min_{z\in{-1,1}^n}z^TQz + b^Tz
$$

Aici, $b$ depinde de $Q$. Reține că pentru a obține $z^TQz + b^Tz$ am eliminat factorul 1/4 și un offset constant de $n^2$, care nu joacă niciun rol în optimizare.

Acum, pentru a obține o formulare cuantică a problemei, promovăm variabilele $z_i$ la o matrice Pauli $Z$, cum ar fi o matrice $2\times 2$ de forma

$$
Z_i = \begin{pmatrix}1 & 0 \\ 0 & -1\end{pmatrix}.
$$

Când substituiești aceste matrice în problema de optimizare de mai sus, obții următorul Hamiltonian

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

*Amintește-ți și că matricele $Z$ sunt încorporate în spațiul computațional al calculatorului cuantic, adică un spațiu Hilbert de dimensiune $2^n\times 2^n$. Prin urmare, termeni precum $Z_iZ_j$ trebuie înțeleși ca produsul tensorial $Z_i\otimes Z_j$ încorporat în spațiul Hilbert $2^n\times 2^n$. De exemplu, într-o problemă cu cinci variabile de decizie, termenul $Z_1Z_3$ se înțelege ca $I\otimes Z_3\otimes I\otimes Z_1\otimes I$, unde $I$ este matricea identitate $2\times 2$.*
</details>

Acest Hamiltonian se numește **Hamiltonianul funcției de cost**. Are proprietatea că starea sa fundamentală corespunde soluției care **minimizează funcția de cost $f(x)$**.
Prin urmare, pentru a rezolva problema ta de optimizare, trebuie acum să pregătești starea fundamentală a lui $H_C$ (sau o stare cu un grad mare de suprapunere cu aceasta) pe calculatorul cuantic. Eșantionând din această stare, cu probabilitate ridicată, vei obține soluția la $min~f(x)$.
Acum să considerăm Hamiltonianul $H_C$ pentru problema **Max-Cut**. Asociem fiecărui vertex al grafului un qubit în starea $|0\rangle$ sau $|1\rangle$, unde valoarea denotă mulțimea din care face parte vertex-ul. Scopul problemei este de a maximiza numărul de muchii $(v_1, v_2)$ pentru care $v_1 = |0\rangle$ și $v_2 = |1\rangle$, sau invers. Dacă asociem operatorul $Z$ cu fiecare qubit, unde

$$
    Z|0\rangle = |0\rangle \qquad Z|1\rangle = -|1\rangle
$$

atunci o muchie $(v_1, v_2)$ face parte din tăietură dacă valoarea proprie a lui $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = -1$; cu alte cuvinte, qubiții asociați lui $v_1$ și $v_2$ sunt diferiți. Similar, $(v_1, v_2)$ nu face parte din tăietură dacă valoarea proprie a lui $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = 1$. Reține că nu ne interesează starea exactă a qubitului asociat fiecărui vertex, ci doar dacă qubiții sunt identici sau diferiți pe o muchie. Problema Max-Cut ne cere să găsim o atribuire a qubiților pe vertex-uri astfel încât valoarea proprie a urmăritorului Hamiltonian să fie minimizată
$$
    H_C = \sum_{(i,j) \in e} Q_{ij} \cdot Z_i Z_j.
$$

Cu alte cuvinte, $b_i = 0$ pentru orice $i$ în problema Max-Cut. Valoarea lui $Q_{ij}$ denotă ponderea muchiei. În acest tutorial considerăm un graf neponderat, adică $Q_{ij} = 1.0$ pentru toți $i, j$.

In [ ]:
def build_max_cut_paulis(
    graph: rx.PyGraph,
) -> list[tuple[str, list[int], float]]:
    """Convert the graph to Pauli list.

    This function does the inverse of `build_max_cut_graph`
    """
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list


max_cut_paulis = build_max_cut_paulis(graph)
cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, n)
print("Cost Function Hamiltonian:", cost_hamiltonian)

Cost Function Hamiltonian: SparsePauliOp(['IIIZZ', 'IIZIZ', 'ZIIIZ', 'IIZZI', 'IZZII', 'ZZIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


#### Hamiltonian &rarr; quantum circuit

The Hamiltonian $H_C$ contains the quantum definition of your problem. Now you can create a quantum circuit that will help *sample* good solutions from the quantum computer. The QAOA is inspired by quantum annealing and applies alternating layers of operators in the quantum circuit.

The general idea is to start in the ground state of a known system, $H^{\otimes n}|0\rangle$ above, and then steer the system into the ground state of the cost operator that you are interested in. This is done by applying the operators $\exp\{-i\gamma_k H_C\}$ and $\exp\{-i\beta_k H_m\}$ with angles $\gamma_1,...,\gamma_p$ and $\beta_1,...,\beta_p~$.


The quantum circuit that you generate is **parametrized** by $\gamma_i$ and $\beta_i$, so you can try out different values of $\gamma_i$ and $\beta_i$ and sample from the resulting state.

![Circuit diagram with QAOA layers](../docs/images/tutorials/quantum-approximate-optimization-algorithm/circuit-diagram.svg)


In this case, you will try an example with one QAOA layer that contains two parameters: $\gamma_1$ and $\beta_1$.

In [4]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=2)
circuit.measure_all()

circuit.draw("mpl")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/7bd8c6d4-f40f-4a11-a440-0b26d9021b53-0.avif" alt="Output of the previous code cell" />

In [5]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1])])

### Step 2: Optimize problem for quantum hardware execution

The circuit above contains a series of abstractions useful to think about quantum algorithms, but not possible to run on the hardware. To be able to run on a QPU, the circuit needs to undergo a series of operations that make up the **transpilation** or **circuit optimization** step of the pattern.

The Qiskit library offers a series of **transpilation passes** that cater to a wide range of circuit transformations. You need to make sure that your circuit is **optimized** for your purpose.

Transpilation may involves several steps, such as:

* **Initial mapping** of the qubits in the circuit (such as decision variables) to physical qubits on the device.
* **Unrolling** of the instructions in the quantum circuit to the hardware-native instructions that the backend understands.
* **Routing** of any qubits in the circuit that interact to physical qubits that are adjacent with one another.
* **Error suppression** by adding single-qubit gates to suppress noise with dynamical decoupling.


More information about transpilation is available in our [documentation](/docs/guides/transpile).

The following code transforms and optimizes the abstract circuit into a format that is ready for execution on one of devices accessible through the cloud using the **Qiskit IBM Runtime service**.

In [6]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
print(backend)

# Create pass manager for transpilation
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit = pm.run(circuit)
candidate_circuit.draw("mpl", fold=False, idle_wires=False)

<IBMBackend('test_heron_pok_1')>


<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

In the QAOA workflow, the optimal QAOA parameters are found in an iterative optimization loop, which runs a series of circuit evaluations and uses a classical optimizer to find the optimal $\beta_k$ and $\gamma_k$ parameters. This execution loop is executed via the following steps:

1. Define the initial parameters
2. Instantiate a new `Session` containing the optimization loop and the primitive used to sample the circuit
3. Once an optimal set of parameters is found, execute the circuit a final time to obtain a final distribution which will be used in the post-process step.

#### Define circuit with initial parameters
We start with arbitrary chosen parameters.

In [7]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

### Pasul 2: Optimizarea problemei pentru execuția pe hardware cuantic
Circuitul de mai sus conține o serie de abstractizări utile pentru a gândi algoritmii cuantici, dar care nu pot fi rulate pe hardware. Pentru a putea rula pe un QPU, circuitul trebuie să treacă printr-o serie de operații care alcătuiesc pasul de **transpilare** sau **optimizare a circuitului** din pattern.

Biblioteca Qiskit oferă o serie de **pase de transpilare** care acoperă o gamă largă de transformări de circuit. Trebuie să te asiguri că circuitul tău este **optimizat** pentru scopul tău.

Transpilarea poate implica mai mulți pași, cum ar fi:

* **Maparea inițială** a qubiților din circuit (cum ar fi variabilele de decizie) la qubiții fizici de pe dispozitiv.
* **Derularea** instrucțiunilor din Circuit cuantic la instrucțiunile native ale hardware-ului pe care le înțelege Backend-ul.
* **Rutarea** oricăror qubiți din circuit care interacționează la qubiți fizici adiacenți.
* **Suprimarea erorilor** prin adăugarea de porți cu un singur qubit pentru a suprima zgomotul cu decuplare dinamică.

Mai multe informații despre transpilare sunt disponibile în [documentația](/guides/transpile) noastră.

Codul următor transformă și optimizează circuitul abstract într-un format gata de execuție pe unul dintre dispozitivele accesibile prin cloud folosind **serviciul Qiskit IBM Runtime**.

In [8]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [9]:
objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -1.6295230263157894
       x: [ 1.530e+00  1.439e+00  4.071e+00  4.434e+00]
    nfev: 26
   maxcv: 0.0


![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif)

### Pasul 3: Execuție folosind primitivele Qiskit
În fluxul de lucru QAOA, parametrii optimi QAOA sunt găsiți într-o buclă iterativă de optimizare, care rulează o serie de evaluări ale circuitului și folosește un optimizator clasic pentru a găsi parametrii optimi $\beta_k$ și $\gamma_k$. Această buclă de execuție este realizată prin următorii pași:

1. Definirea parametrilor inițiali
2. Instanțierea unei noi `Session` care conține bucla de optimizare și primitiva folosită pentru eșantionarea circuitului
3. Odată ce este găsit un set optim de parametri, execuția circuitului o dată în plus pentru a obține o distribuție finală care va fi folosită în pasul de post-procesare.
#### Definirea circuitului cu parametri inițiali
Începem cu parametri aleși arbitrar.

In [10]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif" alt="Output of the previous code cell" />

#### Definirea Backend-ului și a primitivei de execuție
Folosește **primitivele Qiskit Runtime** pentru a interacționa cu Backend-urile IBM&reg;. Cele două primitive sunt Sampler și Estimator, iar alegerea primitivei depinde de tipul de măsurătoare pe care vrei să o rulezi pe calculatorul cuantic. Pentru minimizarea lui $H_C$, folosește Estimator, deoarece măsurarea funcției de cost este pur și simplu valoarea așteptată $\langle H_C \rangle$.
#### Rulare
Primitivele oferă o varietate de [moduri de execuție](/guides/execution-modes) pentru a programa sarcini de lucru pe dispozitivele cuantice, iar un flux de lucru QAOA rulează iterativ într-o sesiune.

![Ilustrare care arată comportamentul modurilor de execuție Single job, Batch și Session.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Poți introduce funcția de cost bazată pe Sampler în rutina de minimizare SciPy pentru a găsi parametrii optimi.

In [11]:
optimized_circuit = candidate_circuit.assign_parameters(result.x)
optimized_circuit.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/2989e76e-4296-4dd8-b065-2b8fced064cf-0.avif" alt="Output of the previous code cell" />

In [12]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

pub = (optimized_circuit,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val / shots for key, val in counts_int.items()}
final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
print(final_distribution_int)

{28: 0.0328, 11: 0.0343, 2: 0.0296, 25: 0.0308, 16: 0.0303, 27: 0.0302, 13: 0.0323, 7: 0.0312, 4: 0.0296, 9: 0.0295, 26: 0.0321, 30: 0.031, 23: 0.0324, 31: 0.0303, 21: 0.0335, 15: 0.0317, 12: 0.0309, 29: 0.0297, 3: 0.0313, 5: 0.0312, 6: 0.0274, 10: 0.0329, 22: 0.0353, 0: 0.0315, 20: 0.0326, 8: 0.0322, 14: 0.0306, 17: 0.0295, 18: 0.0279, 1: 0.0325, 24: 0.0334, 19: 0.0295}


### Step 4: Post-process and return result in desired classical format

The post-processing step interprets the sampling output to return a solution for your original problem. In this case, you are interested in the bitstring with the highest probability as this determines the optimal cut. The symmetries in the problem allow for four possible solutions, and the sampling process will return one of them with a slightly higher probability, but you can see in the plotted distribution below that four of the bitstrings are distinctively more likely than the rest.

In [13]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(graph))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

Result bitstring: [0, 1, 1, 0, 1]


In [14]:
matplotlib.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p[0])].set_color("tab:purple")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif)

Odată ce ai găsit parametrii optimi pentru circuit, poți atribui acești parametri și eșantiona distribuția finală obținută cu parametrii optimizați. Aici trebuie folosită primitiva *Sampler*, deoarece distribuția de probabilitate a măsurătorilor de șiruri de biți corespunde tăieturii optime a grafului.

**Notă:** Aceasta înseamnă pregătirea unei stări cuantice $\psi$ în calculator și apoi măsurarea ei. O măsurătoare va collapsa starea într-o singură stare din baza computațională — de exemplu, `010101110000...` — care corespunde unei soluții candidate $x$ la problema noastră inițială de optimizare ($\max f(x)$ sau $\min f(x)$ în funcție de sarcină).

In [15]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )


plot_result(graph, most_likely_bitstring)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif" alt="Output of the previous code cell" />

And calculate the value of the cut:

In [16]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    assert len(x) == len(
        list(graph.nodes())
    ), "The length of x must coincide with the number of nodes in the graph."
    return sum(
        x[u] * (1 - x[v]) + x[v] * (1 - x[u])
        for u, v in list(graph.edge_list())
    )


cut_value = evaluate_sample(most_likely_bitstring, graph)
print("The value of the cut is:", cut_value)

The value of the cut is: 5


## Part II. Scale it up!

You have access to many devices with over 100 qubits on IBM Quantum&reg; Platform. Select one on which to solve Max-Cut on a 100-node weighted graph. This is a "utility-scale" problem. The steps to build the workflow are followed the same as above, but with a much larger graph.

In [17]:
n = 100  # Number of nodes in graph
graph_100 = rx.PyGraph()
graph_100.add_nodes_from(np.arange(0, n, 1))
elist = []
for edge in backend.coupling_map:
    if edge[0] < n and edge[1] < n:
        elist.append((edge[0], edge[1], 1.0))
graph_100.add_edges_from(elist)
draw_graph(graph_100, node_size=200, with_labels=True, width=1)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif" alt="Output of the previous code cell" />

### Pasul 4: Post-procesare și returnarea rezultatului în formatul clasic dorit
Pasul de post-procesare interpretează ieșirea eșantionării pentru a returna o soluție la problema ta originală. În acest caz, ești interesat de șirul de biți cu probabilitatea cea mai mare, deoarece acesta determină tăietura optimă. Simetriile din problemă permit patru soluții posibile, iar procesul de eșantionare va returna una dintre ele cu o probabilitate ușor mai mare, dar poți vedea în distribuția trasată mai jos că patru dintre șirurile de biți sunt distinctiv mai probabile decât restul.

In [18]:
max_cut_paulis_100 = build_max_cut_paulis(graph_100)

cost_hamiltonian_100 = SparsePauliOp.from_sparse_list(max_cut_paulis_100, 100)
print("Cost Function Hamiltonian:", cost_hamiltonian_100)

Cost Function Hamiltonian: SparsePauliOp(['...'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0

#### Hamiltonian &rarr; quantum circuit

In [19]:
circuit_100 = QAOAAnsatz(cost_operator=cost_hamiltonian_100, reps=1)
circuit_100.measure_all()

circuit_100.draw("mpl", fold=False, scale=0.2, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum execution
To scale the circuit optimization step to utility-scale problems, you can take advantage of the high performance transpilation strategies introduced in Qiskit SDK v1.0. Other tools include the new transpiler service with [AI enhanced transpiler passes](/docs/guides/ai-transpiler-passes).

In [20]:
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit_100 = pm.run(circuit_100)
candidate_circuit_100.draw("mpl", fold=False, scale=0.1, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3a14e7ad-0.avif" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif)

#### Vizualizarea celei mai bune tăieturi
Din șirul de biți optim, poți vizualiza această tăietură pe graful original.

In [21]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_gamma]

objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)

    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit_100, cost_hamiltonian_100, estimator),
        method="COBYLA",
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -3.9939191365979383
       x: [ 1.571e+00  3.142e+00]
    nfev: 29
   maxcv: 0.0


![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif)

Și calculează valoarea tăieturii:

In [22]:
optimized_circuit_100 = candidate_circuit_100.assign_parameters(result.x)
optimized_circuit_100.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/1c432c2e-0.avif" alt="Output of the previous code cell" />

Finally, execute the circuit with the optimal parameters to sample from the corresponding distribution.

In [23]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"


pub = (optimized_circuit_100,)
job = sampler.run([pub], shots=int(1e4))

counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_100_int = {
    key: val / shots for key, val in counts_int.items()
}

## Partea a II-a. Extinde la scară mare!
Ai acces la numeroase dispozitive cu peste 100 de qubiți pe IBM Quantum&reg; Platform. Alege unul pentru a rezolva Max-Cut pe un graf ponderat cu 100 de noduri. Aceasta este o problemă de „scară utilă". Pașii pentru construirea fluxului de lucru sunt urmați la fel ca mai sus, dar cu un graf mult mai mare.

In [24]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/0fda3611-0.avif" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif)

### Pasul 1: Maparea intrărilor clasice într-o problemă cuantică
#### Graf &rarr; Hamiltonian
Mai întâi, convertește graful pe care vrei să-l rezolvi direct într-un Hamiltonian potrivit pentru QAOA.

In [25]:
_PARITY = np.array(
    [-1 if bin(i).count("1") % 2 else 1 for i in range(256)],
    dtype=np.complex128,
)


def evaluate_sparse_pauli(state: int, observable: SparsePauliOp) -> complex:
    """Utility for the evaluation of the expectation value of a measured state."""
    packed_uint8 = np.packbits(observable.paulis.z, axis=1, bitorder="little")
    state_bytes = np.frombuffer(
        state.to_bytes(packed_uint8.shape[1], "little"), dtype=np.uint8
    )
    reduced = np.bitwise_xor.reduce(packed_uint8 & state_bytes, axis=1)
    return np.sum(observable.coeffs * _PARITY[reduced])


def best_solution(samples, hamiltonian):
    """Find solution with lowest cost"""
    min_cost = 1000
    min_sol = None
    for bit_str in samples.keys():
        # Qiskit use little endian hence the [::-1]
        candidate_sol = int(bit_str)
        # fval = qp.objective.evaluate(candidate_sol)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        if fval <= min_cost:
            min_sol = candidate_sol

    return min_sol


best_sol_100 = best_solution(final_distribution_100_int, cost_hamiltonian_100)
best_sol_bitstring_100 = to_bitstring(int(best_sol_100), len(graph_100))
best_sol_bitstring_100.reverse()

print("Result bitstring:", best_sol_bitstring_100)

Result bitstring: [1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1]


Next, visualize the cut. Nodes of the same color belong to the same group.

In [26]:
plot_result(graph_100, best_sol_bitstring_100)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/b4a25e28-0.avif" alt="Output of the previous code cell" />

#### Hamiltonian &rarr; circuit cuantic

In [27]:
cut_value_100 = evaluate_sample(best_sol_bitstring_100, graph_100)
print("The value of the cut is:", cut_value_100)

The value of the cut is: 124


![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif)

### Pasul 2: Optimizarea problemei pentru execuție cuantică
Pentru a extinde pasul de optimizare a circuitului la probleme de scară utilă, poți profita de strategiile de transpilare de înaltă performanță introduse în Qiskit SDK v1.0. Alte instrumente includ noul serviciu de transpilare cu [pasaje de transpilare îmbunătățite cu AI](/guides/ai-transpiler-passes).

In [28]:
# auxiliary function to help plot cumulative distribution functions
def _plot_cdf(objective_values: dict, ax, color):
    x_vals = sorted(objective_values.keys(), reverse=True)
    y_vals = np.cumsum([objective_values[x] for x in x_vals])
    ax.plot(x_vals, y_vals, color=color)


def plot_cdf(dist, ax, title):
    _plot_cdf(
        dist,
        ax,
        "C1",
    )
    ax.vlines(min(list(dist.keys())), 0, 1, "C1", linestyle="--")

    ax.set_title(title)
    ax.set_xlabel("Objective function value")
    ax.set_ylabel("Cumulative distribution function")
    ax.grid(alpha=0.3)


# auxiliary function to convert bit-strings to objective values
def samples_to_objective_values(samples, hamiltonian):
    """Convert the samples to values of the objective function."""

    objective_values = defaultdict(float)
    for bit_str, prob in samples.items():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        objective_values[fval] += prob

    return objective_values

In [29]:
result_dist = samples_to_objective_values(
    final_distribution_100_int, cost_hamiltonian_100
)

Finally, you can plot the cumulative distribution function to visualize how each sample contributes to the total probability distribution and the corresponding objective value. The horizontal spread shows the range of objective values of the samples in the final distribution. Ideally, you would see that the cumulative distribution function has "jumps" at the lower end of the objective function value axis. This would mean that few solutions with low cost have high probability of being sampled. A smooth, wide curve indicates that each sample is similarly likely, and they can have very different objective values, low or high.

In [30]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_cdf(result_dist, ax, "Eagle device")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/4381a2b3-0.avif" alt="Output of the previous code cell" />

Odată ce parametrii optimi obținuți prin rularea QAOA pe dispozitiv au fost găsiți, atribuie parametrii circuitului.